In [1]:
import pandas as pd

DATA_DIR = '/kaggle/input/datasets/abdullahehdayb/olist-abdullah-practice/'

date_cols = ['order_purchase_timestamp', 'order_approved_at',
             'order_delivered_carrier_date', 'order_delivered_customer_date',
             'order_estimated_delivery_date']

train = pd.read_csv(DATA_DIR + 'train.csv', parse_dates=date_cols)
val = pd.read_csv(DATA_DIR + 'val.csv', parse_dates=date_cols)
test = pd.read_csv(DATA_DIR + 'test.csv', parse_dates=date_cols)

print("Train:", train.shape)
print("Val:", val.shape)
print("Test:", test.shape)

Train: (57323, 21)
Val: (20550, 21)
Test: (18603, 21)


In [2]:
def build_features(df):
    out = df.copy()
    out['purchase_month'] = out['order_purchase_timestamp'].dt.month
    out['purchase_weekday'] = out['order_purchase_timestamp'].dt.dayofweek
    out['purchase_hour'] = out['order_purchase_timestamp'].dt.hour
    
    # Days between purchase and estimated delivery - known at purchase time
    out['estimated_days'] = (out['order_estimated_delivery_date'] - out['order_purchase_timestamp']).dt.days
    
    feature_cols = [
        'customer_state', 'purchase_month', 'purchase_weekday', 'purchase_hour',
        'estimated_days', 'n_items', 'total_price', 'total_freight',
        'n_distinct_products', 'n_distinct_sellers', 'total_payment',
        'n_payment_entries', 'max_installments'
    ]
    return out[feature_cols + ['is_late']]

train_feat = build_features(train)
val_feat = build_features(val)
test_feat = build_features(test)

print(train_feat.shape, val_feat.shape, test_feat.shape)
train_feat.head()

(57323, 14) (20550, 14) (18603, 14)


,customer_state,purchase_month,purchase_weekday,purchase_hour,estimated_days,n_items,total_price,total_freight,n_distinct_products,n_distinct_sellers,total_payment,n_payment_entries,max_installments,is_late
0,SP,10,0,10,15,1.0,29.99,8.72,1.0,1.0,38.71,3.0,1.0,0
1,RN,11,5,19,26,1.0,45.00,27.20,1.0,1.0,72.20,1.0,1.0,0
2,SP,2,1,21,12,1.0,19.90,8.72,1.0,1.0,28.62,1.0,1.0,0
3,PR,7,6,21,22,1.0,147.90,27.36,1.0,1.0,175.26,1.0,6.0,0
4,RJ,5,1,13,21,1.0,59.99,15.17,1.0,1.0,75.16,1.0,3.0,0


In [3]:
from sklearn.impute import SimpleImputer

numeric_cols = ['estimated_days', 'n_items', 'total_price', 'total_freight',
                 'n_distinct_products', 'n_distinct_sellers', 'total_payment',
                 'n_payment_entries', 'max_installments']

imputer = SimpleImputer(strategy='median')
imputer.fit(train_feat[numeric_cols])

train_feat[numeric_cols] = imputer.transform(train_feat[numeric_cols])
val_feat[numeric_cols] = imputer.transform(val_feat[numeric_cols])
test_feat[numeric_cols] = imputer.transform(test_feat[numeric_cols])

print("Nulls remaining - train:", train_feat[numeric_cols].isnull().sum().sum())
print("Nulls remaining - val:", val_feat[numeric_cols].isnull().sum().sum())
print("Nulls remaining - test:", test_feat[numeric_cols].isnull().sum().sum())

Nulls remaining - train: 0
Nulls remaining - val: 0
Nulls remaining - test: 0


In [4]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
encoder.fit(train_feat[['customer_state']])

def encode_state(df, encoder):
    encoded = encoder.transform(df[['customer_state']])
    encoded_df = pd.DataFrame(encoded, columns=encoder.get_feature_names_out(['customer_state']), index=df.index)
    return pd.concat([df.drop(columns=['customer_state']), encoded_df], axis=1)

train_final = encode_state(train_feat, encoder)
val_final = encode_state(val_feat, encoder)
test_final = encode_state(test_feat, encoder)

print(train_final.shape, val_final.shape, test_final.shape)
train_final.columns.tolist()

(57323, 40) (20550, 40) (18603, 40)


['purchase_month',
 'purchase_weekday',
 'purchase_hour',
 'estimated_days',
 'n_items',
 'total_price',
 'total_freight',
 'n_distinct_products',
 'n_distinct_sellers',
 'total_payment',
 'n_payment_entries',
 'max_installments',
 'is_late',
 'customer_state_AC',
 'customer_state_AL',
 'customer_state_AM',
 'customer_state_AP',
 'customer_state_BA',
 'customer_state_CE',
 'customer_state_DF',
 'customer_state_ES',
 'customer_state_GO',
 'customer_state_MA',
 'customer_state_MG',
 'customer_state_MS',
 'customer_state_MT',
 'customer_state_PA',
 'customer_state_PB',
 'customer_state_PE',
 'customer_state_PI',
 'customer_state_PR',
 'customer_state_RJ',
 'customer_state_RN',
 'customer_state_RO',
 'customer_state_RR',
 'customer_state_RS',
 'customer_state_SC',
 'customer_state_SE',
 'customer_state_SP',
 'customer_state_TO']

In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(train_final[numeric_cols])

train_final[numeric_cols] = scaler.transform(train_final[numeric_cols])
val_final[numeric_cols] = scaler.transform(val_final[numeric_cols])
test_final[numeric_cols] = scaler.transform(test_final[numeric_cols])

train_final[numeric_cols].describe()

,estimated_days,n_items,total_price,total_freight,n_distinct_products,n_distinct_sellers,total_payment,n_payment_entries,max_installments
count,5.732300e+04,5.732300e+04,5.732300e+04,5.732300e+04,5.732300e+04,5.732300e+04,5.732300e+04,5.732300e+04,5.732300e+04
mean,8.503259e-17,-1.499846e-16,-1.933686e-17,1.175086e-16,-3.728543e-16,-9.420520e-16,8.887517e-17,-7.542614e-17,-1.016425e-16
std,1.000009e+00,1.000009e+00,1.000009e+00,1.000009e+00,1.000009e+00,1.000009e+00,1.000009e+00,1.000009e+00,1.000009e+00
min,-2.210998e+00,-2.621926e-01,-6.442987e-01,-1.135115e+00,-1.704094e-01,-1.026023e-01,-6.837622e-01,-1.206477e-01,-7.204852e-01
25%,-5.818392e-01,-2.621926e-01,-4.330248e-01,-4.057317e-01,-1.704094e-01,-1.026023e-01,-4.444437e-01,-1.206477e-01,-7.204852e-01
50%,-8.055952e-02,-2.621926e-01,-2.436486e-01,-2.795123e-01,-1.704094e-01,-1.026023e-01,-2.495329e-01,-1.206477e-01,-3.598305e-01
75%,5.460400e-01,-2.621926e-01,7.081571e-02,8.569646e-02,-1.704094e-01,-1.026023e-01,7.836871e-02,-1.206477e-01,3.614789e-01
max,1.633635e+01,3.680233e+01,6.445630e+01,5.071263e+01,2.647061e+01,3.516860e+01,6.274126e+01,6.125574e+01,7.574573e+00


In [6]:
import joblib


train_final.to_csv('/kaggle/working/train_features.csv', index=False)
val_final.to_csv('/kaggle/working/val_features.csv', index=False)
test_final.to_csv('/kaggle/working/test_features.csv', index=False)


joblib.dump(imputer, '/kaggle/working/imputer.pkl')
joblib.dump(encoder, '/kaggle/working/encoder.pkl')
joblib.dump(scaler, '/kaggle/working/scaler.pkl')


feature_list = [c for c in train_final.columns if c != 'is_late']
with open('/kaggle/working/feature_list.txt', 'w') as f:
    f.write('\n'.join(feature_list))

print("Saved all artifacts:")
print("- train_features.csv, val_features.csv, test_features.csv")
print("- imputer.pkl, encoder.pkl, scaler.pkl")
print("- feature_list.txt (", len(feature_list), "features )")

Saved all artifacts:
- train_features.csv, val_features.csv, test_features.csv
- imputer.pkl, encoder.pkl, scaler.pkl
- feature_list.txt ( 39 features )
